# Modelo GARCH: Entrenamiento y Evaluación

In [2]:
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.models.lstm_wrapper import LSTMWrapper
from src.eval.scorer import VolatilityScorer
from src.trainer.walk_forward import WalkForwardTrainer

from src.utils import trainer_toolkit as tt
from src.utils import test_toolkit as tk

warnings.filterwarnings("ignore")

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "LSTM"
MODELS_PATH = PROJECT_ROOT / "models"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

print(f"Train: {len(y_train)} obs ({df_train.index[0].date()} → {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs ({df_test.index[0].date()} → {df_test.index[-1].date()})")
print(f"PyTorch {torch.__version__} | dispositivo: CPU")
print(f"Resultados de tablas y gráficas: {RESULTS_PATH}")
print(f"Modelos entrenados:              {MODELS_PATH}")
gpu = f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""
print(f"PyTorch {torch.__version__} | CUDA disponible: {torch.cuda.is_available()}{gpu}")

Train: 2171 obs (2014-01-03 → 2022-11-03)
Test:  953 obs (2022-11-07 → 2026-08-21)
PyTorch 2.5.1+cu121 | dispositivo: CPU
Resultados de tablas y gráficas: c:\Users\User\Desktop\DEEP LEARNING\brent-crude-nlp-market-dynamics\results\LSTM
Modelos entrenados:              c:\Users\User\Desktop\DEEP LEARNING\brent-crude-nlp-market-dynamics\models
PyTorch 2.5.1+cu121 | CUDA disponible: True (NVIDIA GeForce RTX 4060)


c:\Users\User\miniconda3\envs\brent-dl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
import warnings
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.eval.scorer import VolatilityScorer
from src.utils import trainer_toolkit as tt
from src.utils import test_toolkit as tk

try:
    import torch
    from src.models.lstm_wrapper import LSTMWrapper
    from src.trainer.walk_forward import WalkForwardTrainer
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

warnings.filterwarnings("ignore")

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "LSTM"
MODELS_PATH = PROJECT_ROOT / "models"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

print(f"Train: {len(y_train)} obs ({df_train.index[0].date()} → {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs ({df_test.index[0].date()} → {df_test.index[-1].date()})")
if HAS_TORCH:
    assert torch.cuda.is_available(), "CUDA no disponible: revisa la instalación de PyTorch"
    print(f"PyTorch {torch.__version__} | GPU: {torch.cuda.get_device_name(0)}")
else:
    print("PyTorch no instalado: modo solo lectura de resultados")
print(f"Resultados de tablas y gráficas: {RESULTS_PATH}")
print(f"Modelos entrenados:              {MODELS_PATH}")

Train: 2171 obs (2014-01-03 → 2022-11-03)
Test:  953 obs (2022-11-07 → 2026-08-21)
PyTorch no instalado: modo solo lectura de resultados
Resultados de tablas y gráficas: c:\Users\hanne\OneDrive\Desktop\DEEP_LEARNING\brent-crude-nlp-market-dynamics\results\LSTM
Modelos entrenados:              c:\Users\hanne\OneDrive\Desktop\DEEP_LEARNING\brent-crude-nlp-market-dynamics\models


## Introducción

El modelo GARCH (*Generalized Autoregressive Conditional Heteroskedasticity*) permite modelar la dinámica de la volatilidad en series financieras, como los retornos del petróleo Brent. Propuesto por Bollerslev (1986) como una extensión del modelo ARCH de Engle (1982), incorpora tanto los choques pasados como los valores anteriores de la varianza condicional.

El objetivo es entrenar y evaluar el modelo GARCH para estimar la volatilidad de los retornos diarios del Brent, identificar las configuraciones que minimizan la pérdida QLIKE y analizar su capacidad predictiva sobre datos no observados durante el entrenamiento.

### Fundamentos

Los retornos logarítmicos diarios se representan mediante el siguiente proceso estocástico:

$$
r_t = \mu_t + \varepsilon_t
$$

Donde $r_t$ representa el retorno, $\mu_t$ la media condicional y $\varepsilon_t$ la innovación o perturbación aleatoria.

Las innovaciones se expresan como:

$$
\varepsilon_t = \sigma_t z_t
$$

Donde $\sigma_t^2$ es la varianza condicional y $z_t$ es una variable aleatoria estandarizada, con media cero y varianza unitaria.

En las series financieras, la volatilidad no suele ser constante. Se presentan períodos de alta volatilidad seguidos de otros períodos igualmente volátiles, fenómeno conocido como *agrupamiento de volatilidad* (*volatility clustering*).

Además, los retornos del Brent pueden presentar colas pesadas, lo que implica una mayor probabilidad de observar variaciones extremas que las previstas por una distribución normal. Por ello, el modelo permite utilizar distintas distribuciones para las innovaciones, como la Normal, la t de Student y la distribución de error generalizada (GED).

### Modelo ARCH

El modelo GARCH generaliza el modelo ARCH incorporando rezagos de la propia varianza condicional. Su formulación general es:

$$
\sigma_t^2 =
\omega +
\sum_{i=1}^{q} \alpha_i \varepsilon_{t-i}^2 +
\sum_{j=1}^{p} \beta_j \sigma_{t-j}^2
$$

Donde:

- $\omega$: constante que representa el nivel basal de la varianza.
- $\alpha_i$: coeficientes que miden la influencia de los choques pasados al cuadrado.
- $\beta_j$: coeficientes que representan la persistencia de la varianza condicional.
- $q$: orden de los rezagos de las innovaciones al cuadrado.
- $p$: orden de los rezagos de la varianza condicional.

La principal ventaja de GARCH frente a ARCH es que puede representar la persistencia de la volatilidad con menos parámetros, pues incorpora directamente su comportamiento pasado.

Para un modelo GARCH(1,1), la ecuación es:

$$
\sigma_t^2 =
\omega +
\alpha_1 \varepsilon_{t-1}^2 +
\beta_1 \sigma_{t-1}^2
$$

Bajo las condiciones habituales de positividad, la condición de estacionariedad de segundo orden es:

$$
\alpha_1 + \beta_1 < 1
$$

Esta condición permite que la varianza incondicional sea finita y que los efectos de los choques se disipen a largo plazo.

## Diseño Experimental

Se mantiene el mismo diseño experimental utilizado en ARCH para garantizar una comparación justa entre ambos modelos. Si se modificaran los datos, las particiones temporales, las métricas o el procedimiento de validación, sería difícil determinar si las diferencias de desempeño se deben al modelo o a cambios en el experimento.

Por esta razón, se conservan los siguientes elementos:

- **Datos:** los retornos logarítmicos diarios del Brent.
- **Información disponible:** únicamente los retornos históricos, evitando el sesgo de anticipación (*look-ahead bias*).
- **Validación:** cinco particiones temporales mediante validación cruzada de tipo *walk-forward*.
- **Evaluación:** la volatilidad realizada estimada mediante Rogers-Satchell (`rs_daily`) como referencia.
- **Métrica principal:** QLIKE, que permite comparar la calidad de los pronósticos de volatilidad.
- **Distribuciones:** Normal, t de Student y GED.

La búsqueda de hiperparámetros de GARCH explora las combinaciones de $p$ y $q$ entre 1 y 3, junto con las tres distribuciones consideradas. Se realizan cinco ejecuciones para analizar la estabilidad de los resultados.
En definitiva, mantener el diseño de ARCH permite aislar el efecto de incorporar los rezagos de la varianza condicional, que constituye la principal diferencia estructural de GARCH. Así se puede determinar si esta extensión mejora la predicción de la volatilidad del Brent bajo las mismas condiciones experimentales.

## Entrenamiento

Para identificar los mejores hiperparámetros del modelo, se ejecutaron cinco iteraciones independientes del esquema de validación cruzada definido en experimentos anteriores. En cada iteración, se realizó una búsqueda en cuadrícula para los órdenes \(p\) y \(q\), considerando valores entre 1 y 10, junto con tres distribuciones para las innovaciones: Normal, t de Student y GED. El desempeño de las configuraciones se evaluó mediante la métrica QLIKE, con el objetivo de identificar aquella que ofreciera los mejores pronósticos de volatilidad.

In [6]:
scorer = VolatilityScorer()

for exec_id in range(1, 6):
    current_warmup = 1200 + (exec_id * 10)
    exec_dir = RESULTS_PATH / f"exec_{exec_id}"
    exec_dir.mkdir(parents=True, exist_ok=True)

    trainer = WalkForwardTrainer(
        model_class=LSTMWrapper,
        param_grid={
            "window": [5, 10, 22],
            "hidden_size": [16, 32, 64, 128]
        },
        scorer=scorer,
        output_dir=exec_dir,
        n_splits=5,
        warmup=current_warmup
    )

    trainer.run(y_train, rs_train)

Grid Search: 100%|██████████| 12/12 [02:32<00:00, 12.71s/config, params={'hidden_size': 128, 'window': 22}]


In [2]:
frames = []
for exec_id in range(1, 6):
    exec_dir = RESULTS_PATH / f"exec_{exec_id}"
    d = pd.read_csv(exec_dir / "results.csv")
    d["exec"] = exec_id
    frames.append(d)
    ck = json.load(open(exec_dir / "checkpoint.json"))
    print(
        f"exec_{exec_id}: {len(d)} filas (esperado 60) | "
        f"NaN en qlike: {d['qlike'].isna().sum()} | "
        f"{ck['status']} | {ck['timestamp'][:19]} | {ck['total_time_seconds']} s"
    )

df_all = pd.concat(frames, ignore_index=True)

exec_1: 60 filas (esperado 60) | NaN en qlike: 0 | completed | 2026-10-21T00:44:44 | 180.13 s
exec_2: 60 filas (esperado 60) | NaN en qlike: 0 | completed | 2026-10-21T00:47:10 | 145.3 s
exec_3: 60 filas (esperado 60) | NaN en qlike: 0 | completed | 2026-10-21T00:49:38 | 147.95 s
exec_4: 60 filas (esperado 60) | NaN en qlike: 0 | completed | 2026-10-21T00:52:08 | 150.64 s
exec_5: 60 filas (esperado 60) | NaN en qlike: 0 | completed | 2026-10-21T00:54:41 | 152.5 s


Los resultados obtenidos en las cinco réplicas presentan una desviación estándar reducida, lo que evidencia la consistencia del proceso de validación. A partir de los resultados, se observa que la configuración más parsimoniosa, con un orden $(p)$ de 1 y un orden $(q)$ de 1, se sitúa en la primera posición con el menor error global QLIKE, sugiriendo que una menor complejidad es suficiente para capturar adecuadamente la dinámica de la volatilidad en esta fase. Asimismo, la prevalencia de la distribución $t$ de Student en el mejor desempeño es congruente con las colas pesadas y la curtosis elevada documentadas previamente en el EDA.  

In [3]:
tt.display_top_results(
    RESULTS_PATH,
    top_n=5,
    aggregate_mode="mean",
    column_names=(
        "Unidades ocultas (h)", "Ventana (w)",
        "Réplica 1", "Réplica 2", "Réplica 3", "Réplica 4", "Réplica 5",
        "Global (Media ± SD)"
    )
)

Unidades ocultas (h),Ventana (w),Réplica 1,Réplica 2,Réplica 3,Réplica 4,Réplica 5,Global (Media ± SD)
128,5,0.638 ± 0.508,0.531 ± 0.257,0.540 ± 0.332,0.585 ± 0.376,0.588 ± 0.401,0.576 ± 0.043
64,10,0.554 ± 0.308,0.589 ± 0.318,0.562 ± 0.331,0.579 ± 0.353,0.614 ± 0.451,0.580 ± 0.024
128,10,0.601 ± 0.331,0.564 ± 0.228,0.583 ± 0.357,0.581 ± 0.354,0.583 ± 0.344,0.582 ± 0.013
32,10,0.561 ± 0.274,0.600 ± 0.363,0.600 ± 0.352,0.576 ± 0.327,0.589 ± 0.390,0.585 ± 0.017
32,22,0.618 ± 0.346,0.622 ± 0.379,0.587 ± 0.348,0.586 ± 0.361,0.604 ± 0.357,0.603 ± 0.017


Para verificar la robustez de los resultados, la agregación se repitió utilizando la mediana y el rango intercuartílico (IQR). A diferencia del error anterior, se observa una total consistencia con la métrica de la media: el criterio de la mediana también favorece de manera clara a la configuración más parsimoniosa con un menor número de rezagos ($p=1, q=1$) bajo las distribuciones GED, $t$ y Normal. Esto demuestra que, tanto al evaluar el error promedio como el comportamiento central de los días típicos de mercado, el modelo GARCH(1,1) ofrece pronósticos más estables y ajustados, evitando la sobreparametrización de los órdenes superiores.

In [4]:
tt.display_top_results(
    RESULTS_PATH,
    top_n=5,
    aggregate_mode="median",
    column_names=(
        "Unidades ocultas (h)", "Ventana (w)",
        "Réplica 1", "Réplica 2", "Réplica 3", "Réplica 4", "Réplica 5",
        "Global (Mediana ± IQR)"
    )
)

Unidades ocultas (h),Ventana (w),Réplica 1,Réplica 2,Réplica 3,Réplica 4,Réplica 5,Global (Mediana ± IQR)
64,10,0.426 ± 0.142,0.539 ± 0.174,0.418 ± 0.154,0.416 ± 0.174,0.414 ± 0.147,0.418 ± 0.010
64,5,0.432 ± 0.060,0.433 ± 0.100,0.410 ± 0.094,0.442 ± 0.086,0.441 ± 0.103,0.433 ± 0.008
128,5,0.437 ± 0.075,0.439 ± 0.097,0.402 ± 0.088,0.437 ± 0.063,0.415 ± 0.127,0.437 ± 0.023
16,10,0.469 ± 0.068,0.456 ± 0.125,0.449 ± 0.128,0.430 ± 0.119,0.433 ± 0.114,0.449 ± 0.022
32,5,0.460 ± 0.032,0.438 ± 0.081,0.457 ± 0.072,0.448 ± 0.054,0.473 ± 0.106,0.457 ± 0.012


In [6]:
per_exec = (
    df_all.groupby(["hidden_size", "window", "exec"])["qlike"]
          .mean().reset_index()
)
resumen = (
    per_exec.groupby(["hidden_size", "window"])["qlike"]
            .agg(["mean", "std"])
            .sort_values("mean")
            .head(6)
            .round(4)
)
display(resumen)

mean     std
hidden_size window                
128         5       0.5762  0.0429
64          10      0.5798  0.0237
128         10      0.5823  0.0132
32          10      0.5852  0.0168
            22      0.6032  0.0168
64          5       0.6178  0.0259

Dado que los criterios de la media y la mediana coinciden y que GED y $t$ empatan en (1,1), se selecciona la distribución $t$ de Student con $p=1$ y $q=1$. La elección se apoya en las colas pesadas y la curtosis elevada documentadas en el EDA, y en que es la misma distribución seleccionada por el modelo ARCH, lo que permite comparar ambos modelos en la fase de evaluación y atribuir las diferencias de desempeño a la inclusión del rezago de la varianza condicional. Por esta razón, se omite la representación gráfica de la evolución del QLIKE por orden y distribución, pues las tablas anteriores ya permiten identificar la mejor configuración.

Al cuantificar las diferencias entre los órdenes $q=5$ y $q=6$, se observa que la variación en la pérdida QLIKE es marginal, con una diferencia máxima de 0.004 para la distribución Normal y una diferencia nula para la GED (0.428 frente a 0.429). La distribución $t$ de Student demuestra ser la opción más coherente, ya que logra un desempeño sobresaliente (0.420) con un menor número de parámetros. Aunque la distribución Normal ocupa el primer lugar bajo este criterio específico con $q=6$ (0.419), su desempeño general en la fase de validación fue consistentemente inferior, posicionándose como la peor elección en términos de QLIKE al requerir mayor complejidad para aproximarse a los resultados obtenidos por la distribución $t$.

La tabla de estimación muestra que los parámetros de la ecuación de varianza son significativos ($p < .001$). El coeficiente $\omega$ es prácticamente nulo en la escala de los retornos, lo que indica un nivel basal de varianza bajo. El coeficiente $\alpha_1$ ($0.1000$) refleja una reacción moderada de la volatilidad ante los choques recientes, y $\beta_1$ ($0.8800$) una alta persistencia de la varianza condicional. La suma $\alpha_1 + \beta_1 = 0.98$ es menor que 1, por lo que se cumple la condición de estacionariedad en covarianza, aunque está cerca de la unidad: los choques se disipan lentamente, con una vida media de aproximadamente 34 días hábiles. A diferencia del ARCH(5), donde el optimizador repartía el mismo peso ($0.15$) entre los cinco rezagos, GARCH concentra la dinámica en dos coeficientes con valores diferenciados, lo que ilustra su mayor parsimonia. El parámetro $\nu$ ($5.2599$; IC $3.5194\text{–}5.8790$) corresponde a los grados de libertad de la distribución $t$. Un valor bajo y alejado del infinito confirma la presencia de colas pesadas, valida la elección de esta distribución frente a la Normal y coincide con lo identificado en el EDA.

In [ ]:
#Esto hacerlo en un computador con GPU, ya que el entrenamiento de LSTM es costoso y lento en CPU.

assert HAS_TORCH, "Esta celda solo corre en el computador de entrenamiento"

BEST_PARAMS = {"window": 10, "hidden_size": 64}   # <- reemplazar con el ganador

best_model = LSTMWrapper(params={**BEST_PARAMS, "seed": 1})
best_model.fit(y_train)
diag = best_model.get_diagnostics()

print(f"Dispositivo: {best_model.device}")
print(f"Parámetros entrenables: {diag['n_parameters']:,}")
print(f"Mejor época (early stopping): {diag['best_epoch']} de {len(diag['train_losses'])}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(diag["train_losses"], label="Entrenamiento")
ax.plot(diag["val_losses"], label="Validación (early stopping)")
ax.axvline(diag["best_epoch"] - 1, color="gray", ls="--", lw=1)
ax.set_xlabel("Época")
ax.set_ylabel("NLL gaussiana")
ax.set_title("Curva de aprendizaje")
ax.legend()
plt.show()

AssertionError: Esta celda solo corre en el computador de entrenamiento

In [ ]:
assert HAS_TORCH, "Esta celda solo corre en el computador de entrenamiento"

scorer = VolatilityScorer()

rows = []
for s in range(1, 6):
    m = LSTMWrapper(params={**BEST_PARAMS, "seed": s})
    m.fit(y_train)
    p = m.forecast(y_test, initial_context=y_train[-m.context_length:])
    rows.append({"seed": s, **scorer.calculate_all(rs_test, p)})

df_seeds = pd.DataFrame(rows).set_index("seed")
display(df_seeds.round(4))
display(df_seeds.agg(["mean", "std"]).round(4))

In [ ]:
std_residuals = diag["std_resid"]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    column_titles=(
        "Prueba Estadística",
        "Estadístico", "p-valor",
        "p-ajustado (Bonferroni)",
        "Conclusión (α = 0.05)"
    )
)

tt.display_styled_df(
    df=df_diag, decimals=3,
    p_value_cols=["p-valor", "p-ajustado"]
)

In [ ]:
pred_vol_train = diag["conditional_volatility"]

tk.plot_volatility_fit(
    dates=df_train.index,
    rs=rs_train,
    pred_vol=pred_vol_train,
    title="Volatilidad Realizada vs Condicional - LSTM (Entrenamiento)",
    legend_size=13,
    legend_loc="upper left"
)

In [ ]:
scorer = VolatilityScorer()

ctx = best_model.context_length

predictions_test = best_model.forecast(
    y_target=y_test,
    initial_context=y_train[-ctx:]
)

tk.display_train_test_comparison(
    model=best_model,
    scorer=scorer,
    rs_train=rs_train,
    rs_test=rs_test,
    preds_test=predictions_test,
    dist="normal",
    nu=None,
    decimals=6,
    column_titles=(
        "Métrica",
        "Entrenamiento",
        "Prueba",
        "Diferencia"
    ),
    show_diff=True
)

In [ ]:
pred_vol_test = np.sqrt(predictions_test)

tk.plot_volatility_fit(
    dates=df_test.index,
    rs=rs_test,
    pred_vol=pred_vol_test,
    title="Volatilidad Realizada vs Condicional - LSTM (Prueba)",
    legend_loc="upper left",
    legend_size=13
)